# Bloqade native simulator smoke test

Same 2-qubit circuit as `test.ipynb` (H on q0 → CX → X on q1 → Z on q0), but
written as a Bloqade QASM2 kernel and run on Bloqade's built-in
`StackMemorySimulator` (PyQrack-backed).

**Expected:** H + CX produces a Bell state {`|00⟩`, `|11⟩`}, X on q1 flips it
to {`|01⟩`, `|10⟩`}, and Z on q0 adds only an unobservable phase. Repeated
shots should split ~50/50 between `01` and `10`.

In [1]:
from collections import Counter

from bloqade import qasm2
from bloqade.pyqrack import StackMemorySimulator

In [2]:
def draw_bloqade(kernel, mode='text'):
    from qiskit import QuantumCircuit
    from bloqade.qasm2.emit import QASM2
    qc = QuantumCircuit.from_qasm_str(QASM2().emit_str(kernel))
    return qc.draw(output=mode)


## Build the Bloqade kernel

Bloqade uses the `@qasm2.main` decorator to turn a Python function into a
Kirin IR kernel. Inside the function we allocate quantum/classical registers
via `qasm2.qreg(n)` / `qasm2.creg(n)` and apply gates from the `qasm2`
namespace (`qasm2.h`, `qasm2.cx`, etc.).

In [3]:
@qasm2.main
def circ():
    q = qasm2.qreg(2)
    c = qasm2.creg(2)
    qasm2.h(q[0])
    qasm2.cx(q[0], q[1])
    qasm2.x(q[1])
    qasm2.z(q[0])
    qasm2.measure(q, c)
    return c

print('Kernel:')
circ.print()

print(draw_bloqade(circ))                # ASCII
# draw_bloqade(circ, mode='mpl')         # matplotlib figure

Kernel:
func.func @circ() -> !py.CReg {
  ^0(%circ_self):
  │ %0 = qasm2.expr.constant.int 2 : !py.int
  │ %q = qasm2.core.qreg.new(n_qubits=%0) : !py.IList[!py.Qubit, !Any]
  │ %c = qasm2.core.creg.new(n_bits=%0) : !py.CReg
  │ %1 = qasm2.expr.constant.int 0 : !py.int
  │ %2 = qasm2.core.qreg.get(reg=%q, idx=%1) : !py.Qubit
  │      qasm2.uop.h(qarg=%2)
  │ %3 = qasm2.expr.constant.int 1 : !py.int
  │ %4 = qasm2.core.qreg.get(reg=%q, idx=%3) : !py.Qubit
  │      qasm2.uop.CX(ctrl=%2, qarg=%4)
  │      qasm2.uop.x(qarg=%4)
  │      qasm2.uop.z(qarg=%2)
  │      qasm2.core.measure(qarg=%q, carg=%c)
  │      func.return %c
} // func.func circ
     ┌───┐     ┌───┐┌─┐   
q_0: ┤ H ├──■──┤ Z ├┤M├───
     └───┘┌─┴─┐├───┤└╥┘┌─┐
q_1: ─────┤ X ├┤ X ├─╫─┤M├
          └───┘└───┘ ║ └╥┘
c: 2/════════════════╩══╩═
                     0  1 


## Single shot

`sim.run(kernel)` returns a `CRegister` whose entries are
`MeasurementResultValue` enums (0 or 1).

In [4]:
sim = StackMemorySimulator(min_qubits=2)

creg = sim.run(circ)
bits = ''.join(str(int(b)) for b in creg)
print(f'single-shot result (q0 q1): {bits}')

single-shot result (q0 q1): 01


## Many shots → histogram

Call `sim.run(circ)` repeatedly and tally outcomes. Expect the distribution
to concentrate on `01` and `10` in roughly equal proportion.

In [5]:
shots = 1024
counts = Counter()
for _ in range(shots):
    creg = sim.run(circ)
    bits = ''.join(str(int(b)) for b in creg)
    counts[bits] += 1

print(f'{shots} shots:')
for outcome in sorted(counts):
    pct = 100.0 * counts[outcome] / shots
    print(f'  {outcome}: {counts[outcome]:>5d}  ({pct:5.1f}%)')

# Sanity check: only 01 and 10 should appear (up to shot noise).
expected = {'01', '10'}
unexpected = set(counts) - expected
assert not unexpected, f'unexpected outcomes appeared: {unexpected}'
print('\nOK — only expected outcomes (01, 10) observed.')

1024 shots:
  01:   514  ( 50.2%)
  10:   510  ( 49.8%)

OK — only expected outcomes (01, 10) observed.


---

## Send the kernel through `optimize_qasm_bloqade`

`optimize_qasm_bloqade` takes a **QASM file path** (not an in-memory kernel).
So the bridge is:

1. Serialize `circ` to a `.qasm` file using Bloqade's `QASM2` emitter.
2. Call `optimize_qasm_bloqade(path, ...)` with the backend / dialect / passes you want.
3. Use `result.kernel` + `result.backend` downstream.

In [15]:
import sys, os
sys.path.append(os.path.abspath('src'))

from bloqade.qasm2.emit import QASM2
from qasm_transpile_bloqade import optimize_qasm_bloqade

# 1. Serialize the in-memory kernel to a .qasm file
qasm_path = 'circ_from_notebook.qasm'
with open(qasm_path, 'w') as fd:
    fd.write(QASM2().emit_str(circ))

print('Wrote QASM:')
with open(qasm_path) as fd:
    print(fd.read())

Wrote QASM:
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
CX q[0], q[1];
x q[1];
z q[0];
measure q -> c;



In [17]:
# 2. Optimize via the wrapper
result = optimize_qasm_bloqade(
    qasm_path,
    backend_type='stack',
    parallelize=True,
    fold=True,
    verbose=True,
)

print('\n--- ORIGINAL QASM ---')
print(result.original_qasm)
print('--- OPTIMIZED QASM ---')
print(result.transpiled_qasm)
print(f'passes applied: {result.passes_applied}')

Backend: StackMemorySimulator (stack)
  qubits: 2 -> 2
  gates:  4 -> 8
  passes: QASM2Fold, UOpToParallel(native_gates)

--- ORIGINAL QASM ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
h q[0];
CX q[0], q[1];
x q[1];
z q[0];
measure q[0] -> c[0];
measure q[1] -> c[1];

--- OPTIMIZED QASM ---
OPENQASM 2.0;
include "qelib1.inc";
qreg q[2];
creg c[2];
U(1.5707963267949, 0.0, 3.14159265358979) q[0];
U(1.5707963267949, 0.0, 6.28318530717959) q[1];
cz q[0], q[1];
U(1.5707963267949, 3.14159265358979, 3.14159265358979) q[1];
U(0.0, 0.0, 3.14159265358979) q[0];
U(0.0, 0.0, 6.28318530717958) q[1];
U(3.14159265358979, 0.0, 3.14159265358979) q[1];
U(0.0, 0.0, 3.14159265358979) q[0];
measure q[0] -> c[0];
measure q[1] -> c[1];

passes applied: ('QASM2Fold', 'UOpToParallel(native_gates)')


In [10]:
# 3. Run the optimized kernel on the simulator returned by the wrapper.
# Note: QASM-loaded kernels return None (the loader doesn't add `return c`),
# so we drive execution by sampling many shots and tracking the backend's
# internal measurement state via a wrapper kernel.

from collections import Counter
from bloqade import qasm2

loaded_kernel = result.kernel
sim = result.backend

# Drive execution — loaded kernels return None, so we just run and ignore.
# For measurement-outcome histograms, stick with the original `circ` kernel
# (which has `return c`) — the optimized kernel is for inspection / deployment.
shots = 256
for _ in range(shots):
    sim.run(loaded_kernel)
print(f'ran {shots} shots through the optimized kernel (no measurement return).')
print(f'backend: {result.backend_name}')
print(f'gates before/after: {result.original_gates} / {result.transpiled_gates}')

ran 256 shots through the optimized kernel (no measurement return).
backend: StackMemorySimulator
gates before/after: 4 / 8


In [11]:
# 4. (optional) tidy up the file artifact
if os.path.exists(qasm_path):
    os.remove(qasm_path)

---

## An example where `UOpToParallel` actually changes the IR

The earlier example used `h`, `cx`, `x`, `z` — none of which Bloqade's
`UOpToParallel` rule fuses. The rule only groups **`u`, `rz`, `cz`, `barrier`**
because those map directly onto neutral-atom hardware's native parallel
operations (global rotations + simultaneous Rydberg CZ).

To see a real change, build a circuit with `rz` gates on independent qubits.

**Caveat:** the QASM2 *text* emitter re-renders fused `qasm2.parallel.rz`
statements back to individual `rz` lines, so the QASM text looks nearly
identical before/after. The Kirin IR is where the change is visible. This
cell shows both.

In [13]:
from bloqade.qasm2.passes import UOpToParallel
from bloqade.qasm2.emit import QASM2

# Four independent rz rotations — UOpToParallel will fuse these into a single
# qasm2.parallel.rz(qargs=[q0, q1, q2, q3], theta=0.5) statement in the IR.
@qasm2.extended
def parallel_demo():
    q = qasm2.qreg(4)
    qasm2.rz(q[0], 0.5)
    qasm2.rz(q[1], 0.5)
    qasm2.rz(q[2], 0.5)
    qasm2.rz(q[3], 0.5)

print('=== BEFORE UOpToParallel — IR ===')
parallel_demo.print()
print('\n=== BEFORE UOpToParallel — QASM2 text ===')
print(QASM2().emit_str(parallel_demo))

# Apply the pass in-place on the kernel.
UOpToParallel(parallel_demo.dialects)(parallel_demo)

print('\n=== AFTER UOpToParallel — IR ===')
parallel_demo.print()
print('\n=== AFTER UOpToParallel — QASM2 text (note: emitter unfolds parallel ops) ===')
print(QASM2().emit_str(parallel_demo))

=== BEFORE UOpToParallel — IR ===
func.func @parallel_demo() -> !py.NoneType {
  ^0(%parallel_demo_self):
  │  %0 = py.constant.constant 4 : !py.int
  │  %q = qasm2.core.qreg.new(n_qubits=%0) : !py.IList[!py.Qubit, !Any]
  │  %1 = py.constant.constant 0 : !py.int
  │  %2 = qasm2.core.qreg.get(reg=%q, idx=%1) : !py.Qubit
  │  %3 = py.constant.constant 0.5 : !py.float
  │       qasm2.uop.rz(qarg=%2, theta=%3)
  │  %4 = py.constant.constant 1 : !py.int
  │  %5 = qasm2.core.qreg.get(reg=%q, idx=%4) : !py.Qubit
  │  %6 = py.constant.constant 0.5 : !py.float
  │       qasm2.uop.rz(qarg=%5, theta=%6)
  │  %7 = py.constant.constant 2 : !py.int
  │  %8 = qasm2.core.qreg.get(reg=%q, idx=%7) : !py.Qubit
  │  %9 = py.constant.constant 0.5 : !py.float
  │       qasm2.uop.rz(qarg=%8, theta=%9)
  │ %10 = py.constant.constant 3 : !py.int
  │ %11 = qasm2.core.qreg.get(reg=%q, idx=%10) : !py.Qubit
  │ %12 = py.constant.constant 0.5 : !py.float
  │       qasm2.uop.rz(qarg=%11, theta=%12)
  │ %13 = func.c